In [1]:
print("Hello")

Hello


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()

True

In [3]:
llm = ChatOpenAI(
  model=os.getenv("OPENAI_OSS_MODEL"),
  api_key=os.getenv("TOGETHER_API_KEY"),
  base_url=os.getenv("TOGETHER_BASE_URL"),
)

In [4]:
RAG_CHAT_INSTRUCTIONS = """\
You are a subject-matter assistant for a teacher. They will ask you standalone questions - to
understand a concept, get an explanation, or solve a specific problem. There is no paper-
generation intent here; just answer what's asked.

WHAT YOU HAVE
Retrieved chunks from the teacher's own content library (textbook theory, worked examples,
exercises, notes - whatever's indexed), tagged with metadata like subject/grade/chapter where
available. The content may span any grade level, subject, or curriculum - don't assume a
specific one unless the retrieved chunks or the question itself make it clear.

HOW TO ANSWER
- Ground your answer in the retrieved chunks first - use their terminology, method, and
  notation, since that's what matches how the teacher's own material presents the topic.
- If the chunks only partially cover the question (e.g. they explain the concept but the
  teacher asked about an edge case, or asked to solve a problem not in the excerpt), fill the
  gap with your own knowledge - just don't contradict the source material or introduce a
  different method/convention than the one the retrieved content uses for this topic.
- If nothing retrieved is actually relevant, say so and answer from your own knowledge instead
  of forcing a connection to unrelated chunks.
- If retrieved chunks are ambiguous about grade level or method (e.g. a concept taught
  differently across grades/boards), briefly note that rather than silently picking one.

FOR CONCEPT QUESTIONS
Explain clearly and directly, pitched at the level implied by the retrieved content or the
teacher's phrasing. Use the source's own definitions/theorems where possible. A short example
helps more than a long definition - include one if the chunks have one, or make up a simple one
if not.

FOR PROBLEM-SOLVING QUESTIONS
Show the full step-by-step solution, not just the final answer - the teacher likely wants to
check the method or use it for teaching. Match the method/convention the retrieved content uses
for this type of problem unless asked for an alternative approach.

CONFIDENCE
Always give your full answer/findings - never withhold or vaguely dodge a question because
you're unsure. But if you're not confident it's correct (a tricky computation, a thin or
ambiguous source chunk, a topic you're reconstructing mostly from your own knowledge rather
than the retrieved content), say so explicitly: present what you found/worked out, then add a
clear flag such as "Worth double-checking this one" or "I'm not fully certain about this step"
- so the teacher knows to verify before using it, rather than assuming it's textbook-verified.

STYLE
Talk to the teacher as a peer, not a student - be direct, skip motivational framing, skip
"Great question!"-style filler. Keep answers as short as the question allows; expand only when
the question is genuinely open-ended or asks for depth.
"""

human_message = """
Question: {question}
Context: {context}
"""

In [9]:
import sys
from pathlib import Path

# The `app` package lives at the project root (one level up from notebooks/), so
# put that root on sys.path before importing from it.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))




In [12]:
from functools import lru_cache

from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient

from app.core.config import settings


@lru_cache
def get_qdrant_client() -> QdrantClient:
    return QdrantClient(
        url=settings.qdrant_url,
    )


def get_vector_store() -> QdrantVectorStore:
    return QdrantVectorStore(
        client=get_qdrant_client(),
        collection_name=settings.collection_name,
        embedding=OpenAIEmbeddings(
            model=settings.dense_model,
            api_key=settings.openai_api_key,
        ),
        sparse_embedding=FastEmbedSparse(model_name=settings.sparse_model),
        retrieval_mode=RetrievalMode.HYBRID,
    )



In [18]:
vector_store = get_vector_store()

In [22]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [26]:
retrieved_docs = retriever.invoke("How to find HCF of numbers?")
retrieved_docs

[Document(metadata={'book_code': 'jemh1', 'subject': 'Mathematics', 'grade': 10, 'chapter_number': 1, 'chapter_name': 'Real Numbers', 'page_number': 4, 'source_file': 'jemh101.md', 'chunk_id': 'jemh1_ch1_pg4_text', 'chunk_type': 'text', 'content_types': ['theory', 'example'], '_id': '3e4f55a8-bf5e-4d7f-8043-279c81d25ee0', '_collection_name': 'paper-craft'}, page_content='4\n\nMATHEMATICS\n\nIn general, given a composite number $x$, we factorise it as $x = p_1 p_2 \\dots p_n$, where $p_1, p_2, \\dots, p_n$ are primes and written in ascending order, i.e., $p_1 \\leq p_2 \\leq \\dots \\leq p_n$. If we combine the same primes, we will get powers of primes. For example,\n\n$$32760 = 2 \\times 2 \\times 2 \\times 3 \\times 3 \\times 5 \\times 7 \\times 13 = 2^3 \\times 3^2 \\times 5 \\times 7 \\times 13$$\n\nOnce we have decided that the order will be ascending, then the way the number is factorised, is unique.\n\nThe Fundamental Theorem of Arithmetic has many applications, both within mathe

In [46]:
import json

CONTEXT_META_KEYS = ["subject", "grade", "chapter_name", "page_number", "content_types"]


def format_docs(docs):
  blocks = []
  for doc in docs:
    metadata = {k: doc.metadata[k] for k in CONTEXT_META_KEYS if k in doc.metadata}
    blocks.append(
      f"metadata: {json.dumps(metadata, ensure_ascii=False)}\n"
      f"page_content:\n{doc.page_content}"
    )
  return "\n\n------\n\n".join(blocks)

In [45]:
print(format_docs(retrieved_docs))

metadata: {"subject": "Mathematics", "grade": 10, "chapter_name": "Real Numbers", "page_number": 4, "content_types": ["theory", "example"]}
page_content:
4

MATHEMATICS

In general, given a composite number $x$, we factorise it as $x = p_1 p_2 \dots p_n$, where $p_1, p_2, \dots, p_n$ are primes and written in ascending order, i.e., $p_1 \leq p_2 \leq \dots \leq p_n$. If we combine the same primes, we will get powers of primes. For example,

$$32760 = 2 \times 2 \times 2 \times 3 \times 3 \times 5 \times 7 \times 13 = 2^3 \times 3^2 \times 5 \times 7 \times 13$$

Once we have decided that the order will be ascending, then the way the number is factorised, is unique.

The Fundamental Theorem of Arithmetic has many applications, both within mathematics and in other fields. Let us look at some examples.

**Example 1 :** Consider the numbers $4^n$, where $n$ is a natural number. Check whether there is any value of $n$ for which $4^n$ ends with the digit zero.

**Solution :** If the number $

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import  StrOutputParser

chat_prompt = ChatPromptTemplate.from_messages(
  [
    ("system", RAG_CHAT_INSTRUCTIONS),
    ("human", human_message)
    
  ]
)

# chat_prompt.invoke({
#   "question":"WHat is 2+2",
#   "context": "HELLLO"
# })

In [33]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
  {"context": retriever | format_docs, "question": RunnablePassthrough()}
  | chat_prompt
  | llm
  | StrOutputParser()

)


In [41]:
query = "Give me an example of a Polynomial"



In [42]:
for message in rag_chain.stream(query):
  print(message, end="")

A polynomial is an algebraic expression that is a sum of terms of the form \(a x^{n}\) where \(a\) is a real coefficient and \(n\) is a non‑negative integer.  

**Example (quadratic polynomial):**  

\[
p(x)=2x^{2}+3x-5
\]

- Each term has \(x\) raised to a non‑negative integer power (2, 1, 0).  
- The coefficients \(2,\;3,\;-5\) are real numbers.  
- The highest power of \(x\) is 2, so the degree of this polynomial is 2.

You can also use the examples from the textbook, e.g. \(4x+2\) (a linear polynomial, degree 1) or \(5x^{3}-4x^{2}+x-\sqrt{2}\) (a cubic polynomial, degree 3). All satisfy the definition of a polynomial.